# exp_7 · 01 fits — 3 seeds × 28 nested jobs (IDSSE only)
Resumable (one `.npz` + `.pt` per job, cached by fingerprint). Every job predicts all six evaluation views of its
held-out matches with the same weights. Results: `results/fits.json`.

In [1]:
from pathlib import Path
import os, sys, json, time, subprocess
from datetime import datetime, timezone
ROOT = Path.cwd()
if not (ROOT / 'exp7_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_7')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import exp7_common as C
import exp1_common as C1
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
def now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')
def make_log(name):
    path = C.LOG_DIR / f'{name}_progress.log'
    path.parent.mkdir(parents=True, exist_ok=True)
    def log(message, **_):
        print(message, flush=True)
        with path.open('a') as handle:
            handle.write(f'{now()} {message}\n')
    return log
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])

GUARD_FILE = 'guard_01.json'
guard = C.guard_check(C.RESULTS / 'write_guard_before.json', C.RESULTS / GUARD_FILE)
print('nothing written under exp_0-exp_6 / data since freezing:', guard['all_unchanged'],
      {k: v['unchanged'] for k, v in guard['folders'].items()})
if not guard['all_unchanged']:
    raise RuntimeError('Write guard changed: diagnose before continuing')

import exp7_fit as F
log = make_log('01')

PREREG verified 189874eb7d34ab4c | frozen 2026-11-11T01:29:38+00:00


nothing written under exp_0-exp_6 / data since freezing: True {'exp_0': True, 'exp_1': True, 'exp_2': True, 'exp_3': True, 'exp_4': True, 'exp_5': True, 'exp_6': True, 'data': True, 'data_metric_repo': True}


In [2]:
summary = {}
for arm in F.ARMS:
    start = time.time()
    missing = len(F.missing_jobs(arm))
    F.ensure_predictions(arm, log)
    summary[arm] = {'fingerprint': F.fingerprint(arm), 'seed': F.ARMS[arm].seed, 'fitted_now': missing,
                    'seconds': round(time.time() - start, 1), 'jobs_missing_after': len(F.missing_jobs(arm))}
    log(f'ARM {arm} done {summary[arm]}')
count = F.proxy_fit_count()
C1.write_json(C.RESULTS / 'fits.json', {'finished_at_utc': now(), 'arms': summary, 'proxy_fits': count,
                                        'planned': C.PLANNED_FITS, 'cap': C.FIT_CAP})
print('proxy fits:', count)
if count > C.PLANNED_FITS['proxy'] or any(s['jobs_missing_after'] for s in summary.values()):
    raise RuntimeError('Unexpected fit count or missing jobs')

FIT exp7_robust pair_J03WMX_J03WN1 48.0s


FIT exp7_robust pair_J03WMX_J03WOH 47.2s


FIT exp7_robust pair_J03WMX_J03WOY 47.1s


FIT exp7_robust pair_J03WMX_J03WPY 46.8s


FIT exp7_robust pair_J03WMX_J03WQQ 47.2s


FIT exp7_robust pair_J03WMX_J03WR9 46.4s


... (78 outputs left out)

FIT exp7_robust_s2 outer_J03WQQ 53.3s


FIT exp7_robust_s2 outer_J03WR9 53.2s


ARM exp7_robust_s2 done {'fingerprint': 'e5cee02bdc44', 'seed': 20261119, 'fitted_now': 28, 'seconds': 1339.3, 'jobs_missing_after': 0}


proxy fits: 84
